# unit04 レッスン: 分類と評価

**このレッスンで作れるようになるもの**: 「このアヤメはどの品種か」「このワインはどの畑のものか」のように、**ラベル(カテゴリ)を当てる**モデルを学習・評価する処理。さらに「正解率だけ見て安心する」危険を避けるために、**混同行列**で「どう間違えたか」を読み、**交差検証**で「その正解率はたまたまではないか」を検証できるようになります。

unit03 の回帰(数値を当てる)と対になる、機械学習のもう一つの基本形です。しかも使う道具は unit03 とほぼ同じ(`fit` / `predict` / `train_test_split`)。**同じ estimator API のまま、中身のクラスを回帰用から分類用に差し替えるだけ**、という感覚を今日つかみます。

- 所要時間: 15〜25分
- 前提: unit03 を学習済み(`train_test_split` / `fit` / `predict` は既知として進めます)
- 進め方: セルを上から順に実行(`Shift+Enter`)。「書いてみる」セルだけ自分で書く
- 詰まったら: Claude に聞いてOK(答えではなくヒントをくれます)

In [ ]:
import numpy as np
from sklearn.datasets import load_iris, load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    precision_score, recall_score, classification_report,
)

np.set_printoptions(precision=4, suppress=True)   # 表示を見やすく(指数表記を抑制)

def check(name, actual, expected, hint=""):
    try:
        ok = actual is not None and bool(np.all(np.isclose(np.asarray(actual, dtype=float), np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok

import sklearn
print("準備OK! scikit-learn のバージョン:", sklearn.__version__)

---
## 概念1: 分類ワークフロー — 数値ではなくラベルを当てる

### なぜ学ぶか
実務の「予測」の多くは分類です。「このメールは迷惑メールか(はい/いいえ)」「この取引は不正か」「この画像は猫か犬か鳥か」「この顧客は解約するか」。どれも**有限個のカテゴリ(ラベル)のどれかを当てる**問題です。unit03 の回帰(売上金額のような連続値を当てる)と並んで、機械学習の仕事の大半はこの分類が占めます。

### 解説

回帰と分類は**当てる対象の型が違う**だけです。

| | 回帰(unit03) | 分類(今回) |
|---|---|---|
| 当てるもの | 連続値(`double`) | ラベル(`enum`) |
| 例 | 家賃 128,000円 | 品種 = `setosa` |
| 評価 | 誤差の大きさ(MAE/RMSE) | 当たったか外れたか(accuracy) |

決定的に嬉しいのは、**scikit-learn の使い方は回帰と同じ**という点です。C# で言えば `IEstimator` という共通インターフェースがあり、`LinearRegression`(回帰の実装クラス)と `LogisticRegression`(分類の実装クラス)がどちらもそれを満たしている、というイメージ。だから手順は unit03 で覚えた通り:

```
model = LogisticRegression(...)   # ① estimator を生成(実装クラスを差し替えただけ)
model.fit(X_train, y_train)       # ② 学習
pred = model.predict(X_test)      # ③ 予測 → ラベルが返る
```

- **`LogisticRegression`**: 名前に Regression と入っていますが**分類器**です。内部で「各クラスである確率」を計算し、最も確率の高いクラスをラベルとして返します。
- **`accuracy_score(正解, 予測)`**: 予測が正解と一致した割合(0〜1)。「当たった数 ÷ 全体の数」そのものです。

> 補足: `LogisticRegression` は反復計算で解を求めるため、回数が足りないと「収束しなかった」という警告が出ることがあります。今回は `max_iter=1000` のように十分大きく指定して警告を避けます。

次のセルで、`load_iris`(アヤメ150件・3品種の定番データセット)を分類してみます。

In [ ]:
# GOAL: 回帰と同じ fit/predict の手順で、ラベルを当てて正解率を出せることを確認する

# STEP 1: データ読み込み — load_iris().data が特徴量 X、.target が正解ラベル y(0,1,2 の3品種)
iris = load_iris()
X, y = iris.data, iris.target
print("X の形:", X.shape, " y の中身(先頭10件):", y[:10])
print("ラベルの種類:", np.unique(y), "→", iris.target_names)

# STEP 2: 学習用とテスト用に分割 — unit03 と全く同じ関数。random_state=42 で毎回同じ分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# STEP 3: estimator を生成して学習 — LinearRegression が LogisticRegression に変わっただけ
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# STEP 4: 予測 → 数値ではなくラベル(0/1/2)が返る
pred = model.predict(X_test)
print("予測ラベル:", pred)
print("正解ラベル:", y_test)

# STEP 5: 正解率 = 当たった割合
acc = accuracy_score(y_test, pred)
print("accuracy(正解率):", acc)

### 予測してみよう

次のセルは `test_size=0.4`(テスト用を 40% に増やす)で同じことをします。学習に使えるデータが減るので、直感的には「当てにくくなりそう」ですが……

**実行する前に**予測してください: テスト件数は何件になるか(150件の40%)? 正解率は先ほど(1.0)より下がるでしょうか?

In [ ]:
# 予測してから実行!
X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y, test_size=0.4, random_state=42)
model2 = LogisticRegression(max_iter=1000, random_state=42).fit(X_train2, y_train2)
pred2 = model2.predict(X_test2)
print("テスト件数:", len(y_test2))
print("accuracy:", accuracy_score(y_test2, pred2))

アヤメは3品種がきれいに分かれているので、テストを増やしても正解率は 1.0 のまま。「データが減れば必ず悪化する」わけではなく、**問題の易しさ次第**だと分かります(この「簡単すぎる」感覚が、あとで交差検証を学ぶ伏線になります)。

### 書いてみる

**課題**: `test_size=0.5`(半分をテスト)で分割 → `LogisticRegression(max_iter=1000, random_state=42)` を学習 → テストデータを予測し、**「正解した件数」**(正解率ではなく件数そのもの)を `result1` に整数で入れてください。

正解率が `当たった数 ÷ 全体` である以上、まず「当たった数」を数えられることが土台です。予測ラベルと正解ラベルが**一致した個数**を数えます(期待値は整数)。

ヒント(概念レベル): `予測 == 正解` は unit01 のブールマスク。True の個数は `.sum()` で数えられます。

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=42)

result1 = None
# ここに書く(result1 に「正解した件数」を整数で代入する)


check("概念1: 分類ワークフロー", result1, 75,
      hint="model.fit で学習 → model.predict でテストを予測 → (予測 == y_te) の True を .sum() で数える。テストは150の半分=75件")

---
## 概念2: 混同行列と precision / recall — 「どう間違えたか」を見る

### なぜ学ぶか
正解率(accuracy)だけを見ると、**大事故を見逃す**ことがあります。典型例: 工場の不良品検知で、不良品が全体の5%しかないとき、モデルが**何も考えず全部「良品」と答える**だけで accuracy は 95% になります。数字は立派ですが、肝心の不良品は**1個も検出できていない**最悪のモデルです。「全体で何%当たったか」ではなく「**どのクラスを、どのクラスと取り違えたか**」を見ないと、こういう罠に落ちます。医療の陽性見逃し、不正検知の取りこぼしなど、実務ではむしろこちらが本番です。

### 解説

**混同行列(`confusion_matrix(正解, 予測)`)** は、正解ラベルと予測ラベルのクロス集計表です。

> **行 = 正解ラベル、列 = 予測ラベル。**(この向きは必ず言葉で覚えてください。)

`(i, j)` 成分 = 「本当はクラス i なのに、クラス j と予測した件数」。だから:
- **対角線(i = j)= 当たった数**
- **対角線以外 = 取り違えた数**(どこで間違えたかが一目で分かる)

そこから2つの指標が出ます。あるクラスを「陽性」とみなすと:

- **precision(適合率)= 陽性と予測した中で、本当に陽性だった割合**(混同行列の列方向で見る)。C# のテストで言えば「Pass と報告したうち本当に Pass だった割合」= 誤検出の少なさ。
- **recall(再現率)= 本当に陽性のもののうち、陽性と当てられた割合**(行方向で見る)。「本当に Pass のものをどれだけ取りこぼさず検出できたか」= 見逃しの少なさ。

先ほどの「全部良品」モデルは、不良品クラスの **recall が 0**(1個も検出できていない)なので一発で見抜けます。accuracy では見えなかった欠陥です。

- **`precision_score(...)` / `recall_score(...)`**: クラスが3つ以上あるときは `average="macro"` で「各クラスの値の単純平均」を出します。
- **`classification_report(...)`**: クラスごとの precision / recall / f1 を表にした文字列。

次のセルは `load_wine`(ワイン178件・3種)で、**わざと誤りが出る**設定(半分をテストに回す)にして混同行列を読みます。

In [ ]:
# GOAL: 混同行列の対角/非対角を読み、precision と recall が accuracy より詳しいことを確認する

wine = load_wine()
Xw, yw = wine.data, wine.target

# test_size=0.5 でテストを厳しめにする(誤りをわざと発生させて混同行列を読む題材にする)
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(Xw, yw, test_size=0.5, random_state=42)
wine_model = LogisticRegression(max_iter=10000, random_state=42).fit(Xw_tr, yw_tr)
wine_pred = wine_model.predict(Xw_te)

# STEP 1: 全体の正解率 — 一見よさそうな数字
print("accuracy:", round(accuracy_score(yw_te, wine_pred), 4))

# STEP 2: 混同行列 — 行=正解, 列=予測。対角線が的中、それ以外が取り違え
cm = confusion_matrix(yw_te, wine_pred)
print("混同行列(行=正解 / 列=予測):")
print(cm)
print("→ 1行目 [30 3 0]: 本当にクラス0の33件のうち、30件は当たり・3件をクラス1と取り違えた")
print("→ 3行目 [0 0 22]: 本当にクラス2の22件は取り違えゼロ、完璧に当てられている")

# STEP 3: precision と recall(macro=クラスごとの平均)
print("macro precision:", round(precision_score(yw_te, wine_pred, average="macro"), 4))
print("macro recall   :", round(recall_score(yw_te, wine_pred, average="macro"), 4))

# STEP 4: classification_report でクラス別に一覧
print(classification_report(yw_te, wine_pred))

### 予測してみよう

上の混同行列は 3×3 で、対角線の合計が「当たった総数」でした。

**実行する前に**予測してください: 次のセルは `cm` の**対角線の合計**(= 当たった総数)と、**対角線以外の合計**(= 取り違えた総数)を出します。それぞれ何件になりそうですか?(上の混同行列を目で足してみてください。テストは全89件です。)

In [ ]:
# 予測してから実行!
correct = np.trace(cm)              # 対角線の合計 = 的中数
mistakes = cm.sum() - np.trace(cm)  # 残り = 取り違え数
print("的中数(対角線の合計):", correct)
print("取り違え数(それ以外):", mistakes)
print("合計:", correct + mistakes, "= テスト件数")

`np.trace`(対角線の和)は unit01 の集計の応用です。「対角線=正解」を数式でも掴んでおくと、混同行列が一気に読めるようになります。

### 書いてみる

**課題**: 同じ wine データを **`test_size=0.4`(random_state=42)** で分割し直し、`LogisticRegression(max_iter=10000, random_state=42)` を学習 → テストを予測し、その**混同行列**を `result2` に入れてください(行=正解 / 列=予測)。

分割の割合が変わると誤りの出方も変わります。期待値は下のチェックが持っています(3×3 の行列)。

ヒント(概念レベル): 概念1と同じ `fit`→`predict` のあと、`confusion_matrix(正解, 予測)` に渡すだけ。引数の順は (正解, 予測)。

In [ ]:
Xw_tr4, Xw_te4, yw_tr4, yw_te4 = train_test_split(Xw, yw, test_size=0.4, random_state=42)

result2 = None
# ここに書く(result2 に混同行列を代入する)


check("概念2: 混同行列", result2,
      np.array([[26, 0, 0],
                [ 3, 24, 0],
                [ 0, 0, 19]]),
      hint="fit→predict した予測を confusion_matrix(yw_te4, 予測) に渡す。引数は (正解, 予測) の順で行=正解になる")

---
## 概念3: 交差検証 — 「その正解率、たまたまでは?」を潰す

### なぜ学ぶか
`train_test_split` は**1回だけ**データを分けます。すると「たまたま簡単なデータがテストに来た」だけで正解率が高く出ることがあります。この数字を信じてモデルを本番投入すると、実際のデータでは全然当たらない、という失敗が起きます。「1回の測定で結論を出さない」— 実務で性能を報告するときは、この運の要素を潰した数字を出すのが常識です。

### 解説

**交差検証(cross validation)** は、データを K 個の塊に分け、**K 回**「1個をテスト・残りを学習」を回して、K 個の正解率の**平均**を取る方法です。全データが1回ずつテスト側に回るので、1回分割の「引きの良し悪し」が平均でならされます。

C# のテストで言えば、1つのテストケースだけで合否を決めず、複数ケースを回して平均点で判断するイメージ。

- **`cross_val_score(estimator, X, y, cv=5)`**: 「5分割で交差検証して、5個のスコア(正解率)の配列を返す」を1行でやってくれる関数。**未学習**の estimator を渡します(内部で K 回 fit してくれる)。
- 出てきた配列の **平均 = 典型的な性能**、**標準偏差(`.std()`)= 分割によるブレの大きさ**。標準偏差が大きいほど「運に左右されやすい=当てにならない」数字です。

次のセルで、概念1で 1.0 だったアヤメを交差検証してみます。1回分割では見えなかった「ばらつき」が見えます。

In [ ]:
# GOAL: 1回分割の正解率と、交差検証の「平均±ばらつき」の違いを体感する

iris = load_iris()
Xi, yi = iris.data, iris.target

# STEP 1: 未学習の estimator を用意(cross_val_score が内部で fit する)
clf = LogisticRegression(max_iter=1000, random_state=42)

# STEP 2: 5分割交差検証 → 5個のスコアが返る
scores = cross_val_score(clf, Xi, yi, cv=5)
print("5回分のスコア:", scores)

# STEP 3: 平均=典型性能, 標準偏差=ブレの大きさ
print("平均       :", round(scores.mean(), 4))
print("標準偏差   :", round(scores.std(), 4))
print("→ 1回分割では 1.0 に見えたが、分割によっては 0.933 まで下がる回もある。")
print("  平均 0.973 の方が『実力』の正直な数字。")

### 予測してみよう

`cv=5` を `cv=3`(3分割)に変えると、返ってくるスコア配列の**長さは何個**になるでしょう? また、各分割のテストに回るデータ量は増えるでしょうか減るでしょうか?

**実行する前に**予測してください。

In [ ]:
# 予測してから実行!
scores3 = cross_val_score(LogisticRegression(max_iter=1000, random_state=42), Xi, yi, cv=3)
print("cv=3 のスコア:", scores3, " 個数:", len(scores3))
print("平均:", round(scores3.mean(), 4))

`cv=K` ならスコアは K 個(K 回テストするから)。分割数が減るほど1回あたりのテスト量は増えます。「**分割数だけスコアが並ぶ**」と覚えます。

### 書いてみる

**課題**: **wine データ**(`Xw, yw` — 概念2で作った変数がそのまま使えます)を、`LogisticRegression(max_iter=10000, random_state=42)` で **5分割交差検証**し、5個のスコアの**平均**を `result3` に入れてください。

アヤメと違い、ワインは特徴量の桁がバラバラ(スケーリングしていない)なので、平均正解率は 1.0 より下がります。その現実的な数字を出すのが狙いです。

ヒント(概念レベル): `cross_val_score(未学習モデル, Xw, yw, cv=5)` の戻り配列に `.mean()`。

In [ ]:
result3 = None
# ここに書く(result3 に交差検証スコアの平均を代入する)


check("概念3: 交差検証", None if result3 is None else round(float(result3), 4), 0.9611,
      hint="cross_val_score(LogisticRegression(max_iter=10000, random_state=42), Xw, yw, cv=5) の結果に .mean()")

---
## 振り返り(1〜2文でOK — このセルを編集して書き込んでください)

- **今日学んだことを自分の言葉で**:
- **難しかったこと(あれば)**:
- **accuracy だけでは不十分な理由を、一言で**:

(この記述はセッション終了時にチューターが学習ノートとスキルレベル判定に使います)

## まとめと次へ

| 概念 | 一言で | ポイント |
|------|--------|---------|
| 分類ワークフロー | `LogisticRegression` を `fit`→`predict`、`accuracy_score` で評価 | 回帰と同じ API、estimator を差し替えるだけ |
| 混同行列 / precision・recall | `confusion_matrix`(行=正解・列=予測)で「どう間違えたか」を見る | accuracy だけでは「全部良品で95%」の罠を見抜けない |
| 交差検証 | `cross_val_score(model, X, y, cv=5)` の平均±標準偏差 | 1回分割の運を潰し、性能を正直に測る |

**この先どこで使うか**: unit05 の総合プロジェクトでは、今日の分類・評価を **前処理(`StandardScaler` など)と組み合わせて `Pipeline` にまとめます**。今回わざとスケーリングせずにワインの正解率を下げましたが、unit05 でスケーリングを入れると数字が改善するのを体感できます。混同行列と交差検証は、そこでモデルの良し悪しを判断する物差しとして再登場します。

**次**: 演習 `ex01_train_classifier.py` へ。lesson を見ながらで OK。テストは
`python -m pytest courses/ml-intro/unit04-classification/tests/test_ex01.py -q`